# Documentação — Notebook 02_data_load_sqlserver

Este notebook **carrega os dados lidos do ADLS Gen2 para dentro do Azure SQL Server** via JDBC. É o passo que persiste os dados em um banco relacional, tornando-os acessíveis via Lakehouse Federation no Databricks serverless.

**Pré-requisito:** O notebook `01_conexao_adls_gen2` deve ter sido executado (é chamado via `%run`).

## Célula 1 — Execução do notebook anterior

```python
%run ./01_conexao_adls_gen2.ipynb
```

**O que faz:** Executa o notebook `01_conexao_adls_gen2`, que lê os arquivos Parquet do ADLS e cria o PySpark DataFrame `df_clientes`. Após o `%run`, todas as variáveis desse notebook ficam disponíveis aqui.

## Célula 2 — Importação

```python
from dotenv import load_dotenv
import os
```

**O que faz:** Importa `load_dotenv` e `os` para acessar as credenciais do SQL Server armazenadas no `.env`.

## Célula 3 — Carregamento do .env

```python
env_path = next(Path("/Workspace").rglob(".env"), None)
load_dotenv(env_path)
```

**O que faz:** Mesma busca dinâmica do `.env` usada nos outros notebooks. Carrega as credenciais `SQL_HOST`, `SQL_DATABASE`, `SQL_USERNAME` e `SQL_PASSWORD`.

## Célula 4 — Leitura das credenciais SQL

```python
sql_host = os.getenv("SQL_HOST")
sql_database = os.getenv("SQL_DATABASE")
sql_username = os.getenv("SQL_USERNAME")
sql_password = os.getenv("SQL_PASSWORD")
```

**O que faz:** Extrai as 4 credenciais do SQL Server das variáveis de ambiente. Estas credenciais serão usadas para montar a conexão JDBC.

## Célula 5 — Montagem da URL JDBC

```python
jdbc_url = (
    f"jdbc:sqlserver://{sql_host}:1433;"
    f"databaseName={sql_database};"
    "encrypt=true;"
    "trustServerCertificate=true;"
)
```

**O que faz:** Monta a string de conexão JDBC para o SQL Server. A URL inclui:
- Host e porta (1433 — porta padrão do SQL Server)
- Nome do banco de dados
- `encrypt=true` — conexão criptografada
- `trustServerCertificate=true` — confia no certificado do servidor

**Nota:** No Databricks serverless, usa-se `spark.read.format("sqlserver")` com `.option()` ao invés da URL JDBC tradicional, mas a lógica é a mesma.

## Célula 6 — Tabela de destino

```python
tabela_destino = "squad2.ecommerce_clientes"
```

**O que faz:** Define o nome da tabela no SQL Server onde os dados serão gravados. O schema `squad2` é o espaço do Squad 2 no banco `internshipDatabase`.

## Célula 7 — Escrita no SQL Server

```python
df_clientes.write \
    .format("sqlserver") \
    .option("host", sql_host) \
    .option("port", 1433) \
    .option("database", sql_database) \
    .option("dbtable", tabela_destino) \
    .option("user", sql_username) \
    .option("password", sql_password) \
    .mode("overwrite") \
    .save()
```

**O que faz:** Grava o PySpark DataFrame `df_clientes` (criado no notebook 01) no SQL Server usando o conector JDBC do Spark.

**Detalhes importantes:**
- `.mode("overwrite")` — sobrescreve a tabela inteira a cada execução
- `.format("sqlserver")` — usa o driver JDBC do SQL Server nativo do Databricks
- As opções (`host`, `port`, `database`, `dbtable`, `user`, `password`) substituem a URL JDBC tradicional

**⚠️ Cuidado:** `mode("overwrite")` com DataFrame vazio truncaria a tabela. Por isso, o notebook `04_ingestao_bronze` adiciona uma verificação de `count() > 0` antes de escrever.

## Célula 8 — Confirmação (leitura de volta)

```python
df_clientes_sql = spark.read \
    .format("sqlserver") \
    .option("host", sql_host) \
    ... \
    .load()

display(df_clientes_sql)
```

**O que faz:** Lê a tabela recém-gravada do SQL Server de volta para confirmar que os dados foram persistidos corretamente. O `display()` mostra os primeiros registros.

**Resultado esperado:** Tabela com 384 linhas e 8 colunas, mesma estrutura do DataFrame original.

## Resumo do fluxo

| Célula | Função |
| --- | --- |
| 1 | Executa notebook 01 (carrega df_clientes do ADLS) |
| 2 | Importa load_dotenv e os |
| 3 | Carrega .env com credenciais SQL |
| 4 | Lê credenciais SQL do .env |
| 5 | Monta URL JDBC |
| 6 | Define tabela de destino (squad2.ecommerce_clientes) |
| 7 | Escreve dados no SQL Server (mode=overwrite) |
| 8 | Lê de volta para confirmar |

**Fluxo:** Herda dados → carrega credenciais → conecta → escreve → confirma

**Importância:** Este passo torna os dados do ADLS acessíveis no serverless via Lakehouse Federation (catálogo `sqlserver_catalog`).